# 10 Feature Selection

## Objective

Objective:
Identify the most useful features for churn prediction.

This step focuses on:
- Predictive power
- Low redundancy
- Interpretability


## Imports


In [2]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CLEANED_DATA_PATH, TARGET_COLUMN
from src.data.data_loader import load_cleaned_data

sns.set_theme(style="whitegrid")


Matplotlib is building the font cache; this may take a moment.


## Load the Right Dataset

Use the engineered dataset from notebook `09_feature_engineering.ipynb`, not just the cleaned raw file.

This notebook first looks for a saved engineered feature matrix. If that file is not available yet, it falls back to the cleaned dataset so the notebook can still run while the export step is being added to notebook 09.


In [3]:
target_column = TARGET_COLUMN

candidate_engineered_paths = [
    project_root / "data" / "interim" / "telco_churn_engineered.csv",
    project_root / "data" / "processed" / "telco_churn_engineered.csv",
    project_root / "data" / "processed" / "feature_selected_input.csv",
]

engineered_data_path = next(
    (path for path in candidate_engineered_paths if path.exists()),
    None,
)

if engineered_data_path is not None:
    df = pd.read_csv(engineered_data_path)
    dataset_source = f"engineered dataset: {engineered_data_path.relative_to(project_root)}"
else:
    df = load_cleaned_data()
    dataset_source = f"fallback cleaned dataset: {CLEANED_DATA_PATH}"

dataset_summary = {
    "dataset_source": dataset_source,
    "shape": df.shape,
    "target_column": target_column,
    "target_present": target_column in df.columns,
}

dataset_summary


{'dataset_source': 'fallback cleaned dataset: data/interim/telco_churn_cleaned.csv',
 'shape': (7043, 21),
 'target_column': 'Churn',
 'target_present': True}

In [4]:
if target_column not in df.columns:
    raise KeyError(f"Target column '{target_column}' was not found in the loaded dataset.")

df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


The target column for this notebook is confirmed as `Churn` through `TARGET_COLUMN` from the project configuration.

If the notebook loads the cleaned fallback dataset instead of an engineered dataset, update notebook `09_feature_engineering.ipynb` later to export a model-ready feature matrix for the remaining feature-selection steps.
